In [1]:
import pandas as pd
import numpy as np

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df_rt = pd.read_csv("RealTimeData.csv")

df_rt.head()

,Container_ID,Declaration_Date (YYYY-MM-DD),Declaration_Time,Trade_Regime (Import / Export / Transit),Origin_Country,Destination_Port,Destination_Country,HS_Code,Importer_ID,Exporter_ID,Declared_Value,Declared_Weight,Measured_Weight,Shipping_Line,Dwell_Time_Hours,Clearance_Status
0,41256141,2021-04-01,04:38:21,Import,RO,PORT_40,UZ,420231,Y3OM027,FQKH1PD,1248.70,1.0,1.041,LINE_MODE_40,67.6,Clear
1,20889431,2021-04-01,22:33:11,Import,TH,PORT_20,FI,711311,QBIH1EB,543HRUP,1365.35,4.7,4.915,LINE_MODE_10,10.9,Clear
2,25403940,2021-04-01,12:39:00,Import,CN,PORT_40,NO,390690,KEABR8T,69Z0KLV,1105800.00,19000.0,19307.800,LINE_MODE_10,57.4,Clear
3,43489778,2021-04-01,08:37:38,Import,CN,PORT_62,FI,392690,P0E1D1R,LFV52UK,165.00,275.0,287.980,LINE_MODE_40,5.8,Clear
4,94895240,2021-04-01,21:07:20,Import,CN,PORT_40,AM,871390,O4N1E97,JABE2FY,113937.60,42.0,40.580,LINE_MODE_40,18.4,Clear


In [4]:
df_rt.shape

(8481, 16)

In [5]:
df_rt.columns

Index(['Container_ID', 'Declaration_Date (YYYY-MM-DD)', 'Declaration_Time',
       'Trade_Regime (Import / Export / Transit)', 'Origin_Country',
       'Destination_Port', 'Destination_Country', 'HS_Code', 'Importer_ID',
       'Exporter_ID', 'Declared_Value', 'Declared_Weight', 'Measured_Weight',
       'Shipping_Line', 'Dwell_Time_Hours', 'Clearance_Status'],
      dtype='object')

In [6]:
df_rt.isnull().sum()

Container_ID                                0
Declaration_Date (YYYY-MM-DD)               0
Declaration_Time                            0
Trade_Regime (Import / Export / Transit)    0
Origin_Country                              0
Destination_Port                            0
Destination_Country                         0
HS_Code                                     0
Importer_ID                                 0
Exporter_ID                                 0
Declared_Value                              0
Declared_Weight                             0
Measured_Weight                             0
Shipping_Line                               0
Dwell_Time_Hours                            0
Clearance_Status                            0
dtype: int64

In [7]:
df_rt.fillna(0, inplace=True)

In [8]:
df_rt["Declaration_Date (YYYY-MM-DD)"] = pd.to_datetime(df_rt["Declaration_Date (YYYY-MM-DD)"])

In [9]:
df_rt["month"] = df_rt["Declaration_Date (YYYY-MM-DD)"].dt.month
df_rt["day"] = df_rt["Declaration_Date (YYYY-MM-DD)"].dt.day
df_rt["weekday"] = df_rt["Declaration_Date (YYYY-MM-DD)"].dt.weekday

In [10]:
df_rt["hour"] = pd.to_datetime(df_rt["Declaration_Time"]).dt.hour

C:\Users\Pratham\AppData\Local\Temp\ipykernel_7712\382707754.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_rt["hour"] = pd.to_datetime(df_rt["Declaration_Time"]).dt.hour


In [11]:
df_rt.drop(columns=["Declaration_Time"], inplace=True)

In [12]:
df_rt["weight_diff"] = df_rt["Measured_Weight"] - df_rt["Declared_Weight"]

In [13]:
df_rt["weight_ratio"] = df_rt["Measured_Weight"] / (df_rt["Declared_Weight"] + 1)

In [14]:
df_rt["value_per_weight"] = df_rt["Declared_Value"] / (df_rt["Declared_Weight"] + 1)

In [15]:
df_rt["long_dwell"] = (df_rt["Dwell_Time_Hours"] > 48).astype(int)

In [17]:
categorical_cols = [
    "Trade_Regime (Import / Export / Transit)",
    "Origin_Country",
    "Destination_Port",
    "Destination_Country",
    "Shipping_Line"
]

In [18]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in categorical_cols:
    df_rt[col] = le.fit_transform(df_rt[col].astype(str))

In [19]:
container_ids = df_rt["Container_ID"]

In [21]:
df_rt.drop(
    columns=[
        "Container_ID",
        "Importer_ID",
        "Exporter_ID",
        "Declaration_Date (YYYY-MM-DD)",
        "Clearance_Status"
    ],
    inplace=True
)

In [22]:
df_rt.head()

,Trade_Regime (Import / Export / Transit),Origin_Country,Destination_Port,Destination_Country,HS_Code,Declared_Value,Declared_Weight,Measured_Weight,Shipping_Line,Dwell_Time_Hours,month,day,weekday,hour,weight_diff,weight_ratio,value_per_weight,long_dwell
0,0,71,28,14,420231,1248.70,1.0,1.041,54,67.6,4,1,3,4,0.041,0.520500,624.350000,1
1,0,79,21,4,711311,1365.35,4.7,4.915,52,10.9,4,1,3,22,0.215,0.862281,239.535088,0
2,0,17,28,11,390690,1105800.00,19000.0,19307.800,52,57.4,4,1,3,12,307.800,1.016147,58.196937,1
3,0,17,36,4,392690,165.00,275.0,287.980,54,5.8,4,1,3,8,12.980,1.043406,0.597826,0
4,0,17,28,0,871390,113937.60,42.0,40.580,54,18.4,4,1,3,21,-1.420,0.943721,2649.711628,0


In [23]:
df_rt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8481 entries, 0 to 8480
Data columns (total 18 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Trade_Regime (Import / Export / Transit)  8481 non-null   int64  
 1   Origin_Country                            8481 non-null   int64  
 2   Destination_Port                          8481 non-null   int64  
 3   Destination_Country                       8481 non-null   int64  
 4   HS_Code                                   8481 non-null   int64  
 5   Declared_Value                            8481 non-null   float64
 6   Declared_Weight                           8481 non-null   float64
 7   Measured_Weight                           8481 non-null   float64
 8   Shipping_Line                             8481 non-null   int64  
 9   Dwell_Time_Hours                          8481 non-null   float64
 10  month                               

In [24]:
df_rt.shape

(8481, 18)

In [25]:
df_rt.to_csv("processed_realtime_data.csv", index=False)

In [26]:
container_ids.to_csv("container_ids.csv", index=False)